In [1]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, CLIPTokenizer, CLIPTextModel
import transformers.utils.import_utils as import_utils
import_utils.check_torch_load_is_safe = lambda: None

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate project config.py")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config as project_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Device set to  :", device)

PyTorch version: 2.5.1+cu121
Device set to  : cuda


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, CLIPTokenizer, CLIPModel

gpt_tokenizer = AutoTokenizer.from_pretrained(project_config.GPT2_MODEL_NAME)
gpt_model = AutoModelForCausalLM.from_pretrained(project_config.GPT2_MODEL_NAME).to(device)

clip_tokenizer = CLIPTokenizer.from_pretrained(project_config.CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(project_config.CLIP_MODEL_NAME).to(device)

gpt_model.eval()
clip_model.eval()

for param in gpt_model.parameters():
    param.requires_grad = False
for param in clip_model.parameters():
    param.requires_grad = False

print("GPT-2 & CLIP Model loaded and frozen successfully!")

C:\Users\Windows\miniconda3\envs\myenv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
def zerocap_optimize_context(
    context_embeds: torch.Tensor,   # Virtual Context C [1, 5, 768] (requires_grad=True)
    input_ids: torch.Tensor,        # [1, seq_len]
    E_img: torch.Tensor,            # [1, 512]
    n_steps: int = 5,              
    lr: float = 0.05,
    lambda_lang: float = 0.2,
    k: int = 5
) -> torch.Tensor:
    text_embeds = gpt_model.transformer.wte(input_ids)
    # Tính phân phối gốc của GPT-2 khi chưa bị nhiễu bởi gradient
    with torch.no_grad():
        orig_logits = gpt_model(inputs_embeds=text_embeds).logits[:, -1, :]
        orig_probs = F.softmax(orig_logits, dim=-1)

    optimizer = torch.optim.Adam([context_embeds], lr=lr)
    
    for step in range(n_steps):
        optimizer.zero_grad()
        
        combined_embeds = torch.cat([context_embeds, text_embeds], dim=1)
        outputs = gpt_model(inputs_embeds=combined_embeds)
        logits = outputs.logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        
        top_probs, top_indices = torch.topk(probs, k=k, dim=-1)
        
        candidate_texts = [gpt_tokenizer.decode([idx.item()]) for idx in top_indices[0]]
        clip_inputs = clip_tokenizer(candidate_texts, return_tensors="pt", padding=True).to(device)
        text_outputs = clip_model.get_text_features(**clip_inputs)

        if hasattr(text_outputs, "pooler_output"):
            text_features = text_outputs.pooler_output
        elif isinstance(text_outputs, torch.Tensor):
            text_features = text_outputs
        else:
            text_features = text_outputs[0]
            
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # Weighted Sum of Cosine Similarities
        sims = (text_features @ E_img.T).squeeze(-1)
        loss_clip = - (top_probs.squeeze(0) * sims).sum()

        # KL-Divergence giữa phân phối hiện tại và phân phối gốc
        log_probs_full = F.log_softmax(logits, dim=-1)
        loss_lang = F.kl_div(log_probs_full, orig_probs, reduction="batchmean")
        
        total_loss = loss_clip + lambda_lang * loss_lang
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_([context_embeds], max_norm=1.0)
        optimizer.step()
        
    return context_embeds


feature_path = project_config.FEATURE_DIR / "clip_features.pt"

if feature_path.is_file():
    clip_feature_data = torch.load(feature_path, map_location=device)
    real_features = clip_feature_data["features"][:1]
    E_img = real_features / real_features.norm(dim=-1, keepdim=True)
    
    prompt_ids = gpt_tokenizer.encode("Image of a", return_tensors="pt").to(device)
    context_embeds = torch.zeros(1, 5, 768, device=device, requires_grad=True)
    
    print("Testing 1 optimization step with KL-Divergence & Weighted-Sum Cosine...")
    updated_C = zerocap_optimize_context(
        context_embeds=context_embeds,
        input_ids=prompt_ids,
        E_img=E_img,
        n_steps=3
    )
    
    assert updated_C.shape == (1, 5, 768), f"Lỗi shape C: {updated_C.shape}"
    assert torch.isfinite(updated_C).all(), "Phát hiện NaN/Inf trong C"
    assert updated_C.grad is not None, "LỖI: Gradient KHÔNG lan truyền về context_embeds"
    
    print("Sanity check passed")
else:
    print(f"Skipped: Không tìm thấy cache file tại {feature_path.resolve()}")